# 05 — Evaluation

The annual barangay backtest and weekly citywide forecast replay score two different quantities. Annual errors are in percentage points of the barangay case share. Weekly errors are in reported dengue cases.

## Annual barangay allocation backtest

Compare historical barangay shares with equal shares on complete held-out annual observations, using the repository's assumed reporting lag. Annual share errors describe allocation at the annual scale. The source has no observed weekly counts by barangay.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.domain.spatial import annual_backtest
from services.engine.upri_data import load_annual

annual = load_annual()
records = json.loads(annual.to_json(orient="records"))
scores = pd.DataFrame(annual_backtest(records))

evaluation = scores[["targetYear", "weightYear"]].rename(
    columns={"targetYear": "Held-out year", "weightYear": "Weight year"}
)
evaluation["Historical-share MAE (pp)"] = scores["shareMae"] * 100
evaluation["Equal-share MAE (pp)"] = scores["uniformShareMae"] * 100

display(evaluation.round(3))

,Held-out year,Weight year,Historical-share MAE (pp),Equal-share MAE (pp)
0,2019,2017,0.292,0.647
1,2024,2019,0.282,0.671
2,2025,2019,0.327,0.700


## Weekly Quezon City forecast evaluation

This is a retrospective replay over 2024–2025 using the workbook's 52 source blocks per year. The ridge models were refitted at each historical origin using only training outcomes known under the chosen reporting cutoff. Every comparison matches forecast origins, target blocks and horizons.

### Same-week reporting

The initial comparison assumes data through the issue week were available. The table reports mean absolute error (MAE) in cases across 104 target weeks per horizon. An em dash marks a rainfall input unavailable under the relevant reporting rule.

| Weeks ahead | Persistence | Case-only ridge | Ridge +3 rain | Ridge +6 rain |
| --- | --- | --- | --- | --- |
| 1 | 33.60 | 32.50 | 32.10 | 32.53 |
| 2 | 46.47 | 44.19 | 43.31 | 44.20 |
| 3 | 59.27 | 55.42 | 54.37 | 55.43 |
| 4 | 69.71 | 65.89 | — | 65.94 |

### Reporting-delay sensitivity

Each column is the change in MAE after adding rainfall to case-only ridge under that reporting assumption. Negative values favor adding rainfall. Each eligible comparison uses 52 target weeks per year and horizon. Both scenarios assume rainfall arrives one source week late. The combined scenario also delays dengue reports by one source week.

| Target year | Weeks ahead | +3 rain delay only | +3 both delays | +6 rain delay only | +6 both delays |
| --- | --- | --- | --- | --- | --- |
| 2024 | 1 | +0.362 | -0.596 | +0.004 | +0.032 |
| 2024 | 2 | -0.596 | -1.047 | +0.032 | +0.043 |
| 2024 | 3 | — | — | +0.043 | -0.138 |
| 2024 | 4 | — | — | -0.138 | -0.392 |
| 2025 | 1 | -1.174 | -1.164 | +0.037 | -0.020 |
| 2025 | 2 | -1.164 | -1.048 | -0.020 | -0.024 |
| 2025 | 3 | — | — | -0.024 | +0.240 |
| 2025 | 4 | — | — | +0.240 | -0.101 |

Delaying dengue reports alone increased case-only ridge MAE by 7.3–9.1 cases in 2024 and 8.9–14.2 in 2025. With both delays, adding +3 rainfall improved the two-week forecast by about one case in each year.

### High-case weeks with both delays

A week qualifies when its observed count reaches the 90th percentile of counts available at the origin. The threshold uses reports through the preceding week and is a research definition. Bias is the average forecast minus the observed count, so negative values indicate underprediction.

| Target year | Weeks ahead | High-case weeks | Case-only MAE | +3 rain MAE | MAE change | Case-only bias | +3 rain bias |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2024 | 1 | 17 | 67.20 | 67.30 | +0.10 | -52.11 | -51.60 |
| 2024 | 2 | 17 | 80.66 | 79.18 | -1.49 | -73.45 | -72.78 |
| 2025 | 1 | 19 | 94.66 | 92.66 | -2.00 | -54.67 | -52.61 |
| 2025 | 2 | 19 | 110.64 | 108.06 | -2.58 | -79.70 | -77.30 |

### Interpretation and reproducibility

**Rainfall availability.** The workbook's RF_NASA uses NASA IMERG Final Run V07. NASA lists about 3.5 months of latency for the daily Final product, so the tested one-week rainfall delay is hypothetical. The +3 and +6 rainfall inputs would typically be unavailable at the replayed forecast origins if using Final Run. An operational comparison needs a separately validated low-latency rainfall product such as IMERG Early or Late, aligned to the QC source weeks. [Workbook source notes](../data/raw/upri_noah/README.md) · [NASA Final Run product documentation](https://disc.gsfc.nasa.gov/datasets/GPM_3IMERGDF_07/summary) · [NASA IMERG products](https://gpm.nasa.gov/data/imerg)

- Two weeks ahead is the most consistent +3 rainfall result across the tested delay assumptions. Substantial underprediction remains in high-case weeks.
- Earlier exploratory analysis used both evaluation years when identifying candidate lags. Actual source publication times are unknown. Confirmation would require later unseen targets and verified reporting times. The analysis estimates point forecast error. Annual share validation applies at the barangay-year scale.
- Reproduce the weekly tables with the repository Python environment by opening [04 — Model training](04_model_training.ipynb) and choosing **Restart Kernel and Run All**. The leakage cell rereads the saved training code, so save notebook 04 before running it. The annual comparison above runs directly in notebook 05 using the existing annual backtest. See [01 — Standardization](01_standardization.ipynb) for the source audit.
